<a href="https://colab.research.google.com/github/dehigher/ai-practice/blob/main/2_%E6%8F%90%E7%A4%BA%E8%AF%8D%E6%8A%80%E5%B7%A7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 提示词技巧
1. 反向提示词，不使用...
2. 规范输出格式： 输出json、输出sql，便于程序使用
3. 异常输出，code、message
4. 尽量使用英文
5. 不断迭代

In [8]:
!pip3 install openai

## 意图识别

In [1]:
from openai import OpenAI


client = OpenAI(api_key = 'sk-3547bbf4b0da4f8eb5a99a049e2614d4', base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-reasoner",
    temperature=0,
    stream=False,
    messages=[
        {
            "role":"system",
            "content": '''Recognize the intent from the user's input '''
        },
        {
            "role":"user",
            "content":"订明天早5点北京到上海的飞机"
        }
    ]
)


print(response.choices[0].message.content)


根据您的输入，我理解您希望预订一张从北京到上海的机票，出发时间定为明天早上5点左右。为了准确为您查询航班，我需要确认一些细节：

1. **具体日期**：您提到的“明天”是指哪一天？请提供具体日期，以便我查询准确的航班信息。
2. **时间范围**：“早5点”是指出发时间吗？如果是，是否需要精确到5:00整，还是允许前后一段时间（例如4:00-6:00之间）？
3. **其他信息**：请问需要几张机票？以及您对航空公司、舱位等级（如经济舱、商务舱）有特殊要求吗？

请补充这些信息，我将立即为您查找合适的航班选项。


### 规范输出 - 输出json格式

In [6]:
from openai import OpenAI


client = OpenAI(api_key = 'sk-3547bbf4b0da4f8eb5a99a049e2614d4', base_url="https://api.deepseek.com")

system_prompt = """ Recognize the intent from the user's input and format output as JSON string.
        The output JSON string includes: "intention", "paramters" """

response = client.chat.completions.create(
    model="deepseek-reasoner",
    temperature=0,
    stream=False,
    messages=[
        {
            "role":"system",
            "content": system_prompt
        },
        {
            "role":"user",
            "content":"订明天早5点北京到上海的飞机"
        }
    ]
)


print(response.choices[0].message.content)

{
  "intention": "book_flight",
  "parameters": {
    "departure_city": "北京",
    "arrival_city": "上海",
    "date": "明天",
    "time": "05:00"
  }
}


## 生成SQL

In [5]:
from openai import OpenAI


client = OpenAI(api_key = 'sk-3547bbf4b0da4f8eb5a99a049e2614d4', base_url="https://api.deepseek.com")


system_prompt =  """ You are a software engineer, you can anwser the user request based on the given tables:
                  table “students“ with the columns [id, name, course_id, score]
                  table "courses" with the columns [id, name]
                  """

response = client.chat.completions.create(
    model="deepseek-chat",
    temperature=1,
    stream=False,
    messages=[
        {
          'role': 'system',
          'content': system_prompt
        },
        {
          'role': 'user',
          'content': '计算所有学生英语课程的平均成绩'
        }
    ],
    max_tokens = 500
)
print(response.choices[0].message.content)

要计算所有学生英语课程的平均成绩，可以按以下步骤进行：

1. 首先，在 `courses` 表中找到课程名为“英语”的课程ID。
2. 然后，在 `students` 表中筛选出 `course_id` 等于该课程ID的记录。
3. 最后，计算这些学生成绩的平均值。

相应的 SQL 查询如下：

```sql
SELECT AVG(s.score) AS average_score
FROM students s
JOIN courses c ON s.course_id = c.id
WHERE c.name = '英语';
```

如果 `courses` 表中没有名为“英语”的课程，或者没有学生选修该课程，查询将返回 `NULL`。


### 规范输出 - 只输出SQL

In [7]:
from openai import OpenAI


client = OpenAI(api_key = 'sk-3547bbf4b0da4f8eb5a99a049e2614d4', base_url="https://api.deepseek.com")


system_prompt =  """ You are a software engineer, you can anwser the user request based on the given tables:
                  table “students“ with the columns [id, name, course_id, score]
                  table "courses" with the columns [id, name]
                  """

response = client.chat.completions.create(
    model="deepseek-chat",
    temperature=1,
    stream=False,
    messages=[
        {
          'role': 'system',
          'content': system_prompt
        },
        {
          'role': 'user',
          'content': '计算所有学生英语课程的平均成绩，返回结果只包括SQL'
        }
    ],
    max_tokens = 500
)
print(response.choices[0].message.content)

```sql
SELECT AVG(s.score) AS average_score
FROM students s
JOIN courses c ON s.course_id = c.id
WHERE c.name = '英语';
```
